# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ali-Haider987/alihaider-flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Contract — lane: Content Decline / Refresh Prioritization

*(Skills loaded for this notebook: `writing-data-contracts`, `querying-big-datasets`, and `flyrank/flyrank-data` per `skills/README.md`.)*

**One row (unit of analysis) = one content item, for one client, summarized over one month.**
The warehouse's native grain — per the dataset card — is **"one row per report date, pseudonymized client, and pseudonymized content item"** (`fact_content_daily_performance`, 78,835,655 rows total across the full release). I aggregate that daily grain up to a monthly content-level row here, because the decision I'm supporting — *"should this page be queued for refresh?"* — is made per page, not per page-per-day.

**Tables used** (grain as documented on the dataset card):
- `fact_content_daily_performance` — one row per (report date, client, content item). Primary source for features and the label. Partitioned by `month=YYYY-MM`.
- `fact_content_query_90d` — one row per (client, content item, **query hash**) over a fixed trailing 90-day window. I use it for content-level query-mix signals only (via `ANY_VALUE`, since those aggregates are repeated across every query row for a given content item).
- `dim_content` — one row per content item. Context + one static feature (word count).
- `dim_clients` — one row per client. Used only to check GA4-export availability, never fed to the model.

**Time window:** a single mid-panel month, **`2026-03`** (not the sealed final month `2026-06`, which the warehouse's own README flags as the natural outcome window of any past→future label and reserves as held-out). Within that month I split on the 15th:
- **Decision moment:** end of day, 2026-03-15.
- **Feature window:** 2026-03-01 → 2026-03-15 ("first half") — fully observed *before* the decision moment.
- **Outcome window:** 2026-03-16 → 2026-03-31 ("second half") — the future relative to the decision moment; this is where the label comes from.

**Label / proxy:** `is_declining` = 1 if second-half GSC impressions fall below 80% of first-half GSC impressions. It's an *observed outcome* built from raw counts I compute myself — not one of FlyRank's own precomputed `trend_*` / `health_score` fields (those live in the separate `internship-lanes` dataset, and FlyRank's own card for that dataset explicitly flags them as **leakage-risk context, not default features**). Building my own label from raw impressions avoids inheriting that risk.

**One thing I deliberately exclude:** raw GA4 event-level rows and individual (non-aggregated) query strings. I only use pre-aggregated daily/90-day summaries. Event- and query-level detail is finer-grained than my unit of analysis needs, and query strings can carry near-identifying long-tail phrasing I don't need to touch for this lane.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Setup: connect DuckDB to the hosted release (same pattern as notebook 03).
%pip -q install duckdb huggingface_hub

import os, getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Verification: confirm every table exists and see rough scale before claiming anything about it.
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

# Discover real column names rather than assuming them — this is the "verify, don't guess" habit
# the whole contract depends on.
def columns_of(table_sql):
    return [r[0] for r in con.sql(f'DESCRIBE SELECT * FROM {table_sql} LIMIT 0').fetchall()]

def resolve_col(table_sql, candidates):
    cols = columns_of(table_sql)
    for c in candidates:
        if c in cols:
            return c
    raise ValueError(f'None of {candidates} found. Actual columns: {cols}')

print('\nfact_daily columns:   ', columns_of(TABLES['fact_daily']))
print('dim_content columns:  ', columns_of(TABLES['dim_content']))
print('dim_clients columns:  ', columns_of(TABLES['dim_clients']))
print('fact_query_90d columns:', columns_of(TABLES['fact_query_90d']))


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.